# COE Price Prediction & Quota Elasticity (Category A & B)

Predicts COE premiums for Category A and B, then simulates how adding quota would move predicted premiums.

## Approach

1. Load COE bidding results.
2. Filter Category A and B.
3. Create demand-pressure and time-series features.
4. Train baseline and tree-based models using a time-based holdout split.
5. Select the best model by RMSE.
6. Simulate quota increments and estimate model-based elasticity.

In [1]:
"""
COE Price Prediction & Quota Elasticity Analysis for Category A and B
Zhang Jie

Expected input files in a `data/` folder next to this notebook:
    - COEBiddingResultsPrices.csv
    - MotorVehicleQuotaQuotaPremiumAndPrevailingQuotaPremiumMonthly.csv (optional, not required)

Outputs:
    - outputs/model_metrics.csv
    - outputs/policy_simulation.csv
    - outputs/*.png charts
"""

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def rmse_score(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


RANDOM_STATE = 42


def find_file(filename: str) -> Path:
    candidates = [
        Path(filename),
        Path("data") / filename,
        Path("..") / filename,
        Path("../..") / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Please place it next to this notebook or in a data/ subfolder.")


def clean_int_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s.astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")


def load_and_clean() -> pd.DataFrame:
    path = find_file("COEBiddingResultsPrices.csv")
    df = pd.read_csv(path)

    # Keep Category A and B only, as required by the question.
    df = df[df["vehicle_class"].isin(["Category A", "Category B"])].copy()

    for col in ["quota", "bids_success", "bids_received", "premium"]:
        df[col] = clean_int_series(df[col])

    df["month_dt"] = pd.to_datetime(df["month"] + "-01")
    df["auction_index"] = df["month_dt"].rank(method="dense").astype(int) * 10 + df["bidding_no"]
    df = df.sort_values(["month_dt", "bidding_no", "vehicle_class"]).reset_index(drop=True)

    # Basic pressure indicators.
    df["bid_to_quota"] = df["bids_received"] / df["quota"]
    df["quota_utilisation"] = df["bids_success"] / df["quota"]
    df["unsuccessful_bid_rate"] = (df["bids_received"] - df["bids_success"]) / df["bids_received"]
    df["year"] = df["month_dt"].dt.year
    df["month_num"] = df["month_dt"].dt.month
    df["quarter"] = df["month_dt"].dt.quarter
    df["post_covid"] = (df["month_dt"] >= "2020-04-01").astype(int)
    df["post_2022_high_price_regime"] = (df["month_dt"] >= "2022-01-01").astype(int)

    # Time-series features should only use past information.
    g = df.groupby("vehicle_class", group_keys=False)
    for lag in [1, 2, 3, 6]:
        df[f"premium_lag_{lag}"] = g["premium"].shift(lag)
        df[f"quota_lag_{lag}"] = g["quota"].shift(lag)
        df[f"bid_to_quota_lag_{lag}"] = g["bid_to_quota"].shift(lag)

    for window in [3, 6, 12]:
        df[f"premium_roll_mean_{window}"] = g["premium"].transform(lambda x: x.shift(1).rolling(window).mean())
        df[f"premium_roll_std_{window}"] = g["premium"].transform(lambda x: x.shift(1).rolling(window).std())
        df[f"quota_roll_mean_{window}"] = g["quota"].transform(lambda x: x.shift(1).rolling(window).mean())

    df["quota_change_pct"] = g["quota"].pct_change()
    df["premium_change_pct_lag_1"] = g["premium"].transform(lambda x: x.pct_change().shift(1))

    df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return df


def get_feature_columns():
    numeric_features = [
        "quota", "bids_success", "bids_received", "bid_to_quota", "quota_utilisation", "unsuccessful_bid_rate",
        "bidding_no", "year", "month_num", "quarter", "post_covid", "post_2022_high_price_regime",
        "premium_lag_1", "premium_lag_2", "premium_lag_3", "premium_lag_6",
        "quota_lag_1", "quota_lag_2", "quota_lag_3", "quota_lag_6",
        "bid_to_quota_lag_1", "bid_to_quota_lag_2", "bid_to_quota_lag_3", "bid_to_quota_lag_6",
        "premium_roll_mean_3", "premium_roll_mean_6", "premium_roll_mean_12",
        "premium_roll_std_3", "premium_roll_std_6", "premium_roll_std_12",
        "quota_roll_mean_3", "quota_roll_mean_6", "quota_roll_mean_12",
        "quota_change_pct", "premium_change_pct_lag_1",
    ]
    categorical_features = ["vehicle_class"]
    return numeric_features, categorical_features


def make_preprocessor(numeric_features, categorical_features):
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer([
        ("num", StandardScaler(), numeric_features),
        ("cat", ohe, categorical_features),
    ])


def train_models(df: pd.DataFrame):
    numeric_features, categorical_features = get_feature_columns()
    features = numeric_features + categorical_features

    # Time-based holdout: last 20% of auction rounds is test data.
    unique_rounds = np.sort(df["auction_index"].unique())
    cutoff = unique_rounds[int(len(unique_rounds) * 0.8)]
    train_df = df[df["auction_index"] < cutoff].copy()
    test_df = df[df["auction_index"] >= cutoff].copy()

    X_train, y_train = train_df[features], train_df["premium"]
    X_test, y_test = test_df[features], test_df["premium"]

    preprocessor = make_preprocessor(numeric_features, categorical_features)

    models = {
        "Ridge baseline": Ridge(alpha=1.0),
        "Random Forest": RandomForestRegressor(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=4,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            random_state=RANDOM_STATE,
        ),
    }

    fitted = {}
    metric_rows = []

    for name, model in models.items():
        pipe = Pipeline([("prep", preprocessor), ("model", model)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)

        metric_rows.append({
            "model": name,
            "MAE": mean_absolute_error(y_test, pred),
            "RMSE": rmse_score(y_test, pred),
            "R2": r2_score(y_test, pred),
            "MAPE": np.mean(np.abs((y_test - pred) / y_test)) * 100,
        })

        for cat in ["Category A", "Category B"]:
            mask = test_df["vehicle_class"] == cat
            if mask.sum() > 0:
                metric_rows.append({
                    "model": f"{name} - {cat}",
                    "MAE": mean_absolute_error(y_test[mask], pred[mask]),
                    "RMSE": rmse_score(y_test[mask], pred[mask]),
                    "R2": r2_score(y_test[mask], pred[mask]),
                    "MAPE": np.mean(np.abs((y_test[mask] - pred[mask]) / y_test[mask])) * 100,
                })

        fitted[name] = pipe

    metrics = pd.DataFrame(metric_rows).sort_values("RMSE")
    best_name = metrics[~metrics["model"].str.contains(" - ")].iloc[0]["model"]
    best_model = fitted[best_name]

    test_out = test_df[["month", "bidding_no", "vehicle_class", "premium", "quota", "bids_received", "bid_to_quota"]].copy()
    test_out["predicted_premium"] = best_model.predict(X_test)
    test_out["error"] = test_out["predicted_premium"] - test_out["premium"]

    return best_name, best_model, metrics, train_df, test_df, test_out, features


def run_policy_simulation(df: pd.DataFrame, model, features):
    """Simulate adding quota while holding demand fixed.

    Assumption: bids_received stays put in the short run, so extra quota just
    lowers the bid-to-quota ratio. Not a causal estimate - a policy sensitivity
    check based on how the model responds to that assumption.
    """
    increments = [0, 50, 100, 200, 300, 500, 800]
    rows = []

    latest_rows = df.sort_values(["month_dt", "bidding_no"]).groupby("vehicle_class").tail(1)

    for _, base in latest_rows.iterrows():
        base_input = base.copy()
        base_price = None
        for inc in increments:
            scenario = base_input.copy()
            original_quota = float(base_input["quota"])
            new_quota = original_quota + inc
            bids_received = float(base_input["bids_received"])
            bids_success = min(new_quota, bids_received)

            scenario["quota"] = new_quota
            scenario["bids_success"] = bids_success
            scenario["bid_to_quota"] = bids_received / new_quota
            scenario["quota_utilisation"] = bids_success / new_quota
            scenario["unsuccessful_bid_rate"] = max(bids_received - bids_success, 0) / bids_received
            scenario["quota_change_pct"] = (new_quota - float(base_input["quota_lag_1"])) / float(base_input["quota_lag_1"])

            X = pd.DataFrame([scenario[features]])
            pred = float(model.predict(X)[0])
            if base_price is None:
                base_price = pred

            pct_quota_change = inc / original_quota if original_quota else np.nan
            pct_price_change = (pred - base_price) / base_price if base_price else np.nan
            elasticity = pct_price_change / pct_quota_change if inc > 0 and pct_quota_change != 0 else np.nan

            rows.append({
                "vehicle_class": base_input["vehicle_class"],
                "base_month": base_input["month"],
                "base_bidding_no": int(base_input["bidding_no"]),
                "quota_increment": inc,
                "scenario_quota": int(new_quota),
                "assumed_bids_received": int(bids_received),
                "scenario_bid_to_quota": scenario["bid_to_quota"],
                "predicted_premium": pred,
                "price_change_vs_base": pred - base_price,
                "price_change_pct_vs_base": pct_price_change * 100,
                "quota_change_pct_vs_base": pct_quota_change * 100,
                "model_based_elasticity": elasticity,
            })

    return pd.DataFrame(rows)


def plot_outputs(df, test_out, sim, metrics, outdir: Path):
    outdir.mkdir(parents=True, exist_ok=True)

    # 1. Price trend.
    plt.figure(figsize=(11, 6))
    for cat, d in df.groupby("vehicle_class"):
        d = d.sort_values(["month_dt", "bidding_no"])
        plt.plot(d["month_dt"], d["premium"], label=cat)
    plt.title("COE premiums for Category A and B have climbed sharply")
    plt.ylabel("COE premium (S$)")
    plt.xlabel("Auction month")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outdir / "01_price_trend.png", dpi=200)
    plt.close()

    # 2. Demand pressure relationship.
    plt.figure(figsize=(9, 6))
    for cat, d in df.groupby("vehicle_class"):
        plt.scatter(d["bid_to_quota"], d["premium"], alpha=0.6, label=cat)
    plt.title("Demand pressure is strongly associated with COE premiums")
    plt.xlabel("Bid-to-quota ratio")
    plt.ylabel("COE premium (S$)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outdir / "02_bid_to_quota_vs_price.png", dpi=200)
    plt.close()

    # 3. Actual vs predicted.
    plt.figure(figsize=(11, 6))
    for cat, d in test_out.groupby("vehicle_class"):
        d = d.sort_values(["month", "bidding_no"]).copy()
        d["period"] = d["month"] + " #" + d["bidding_no"].astype(str)
        plt.plot(range(len(d)), d["premium"], label=f"Actual {cat}")
        plt.plot(range(len(d)), d["predicted_premium"], linestyle="--", label=f"Predicted {cat}")
    plt.title("Model tracks broad COE price movements in the holdout period")
    plt.ylabel("COE premium (S$)")
    plt.xlabel("Holdout auction sequence")
    plt.legend(ncol=2)
    plt.tight_layout()
    plt.savefig(outdir / "03_actual_vs_predicted.png", dpi=200)
    plt.close()

    # 4. Policy simulation.
    plt.figure(figsize=(9, 6))
    for cat, d in sim.groupby("vehicle_class"):
        plt.plot(d["quota_increment"], d["predicted_premium"], marker="o", label=cat)
    plt.title("Policy simulator: predicted premium under additional quota")
    plt.xlabel("Additional quota certificates")
    plt.ylabel("Predicted COE premium (S$)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outdir / "04_policy_simulation.png", dpi=200)
    plt.close()

    # 5. Metrics bar chart.
    main_metrics = metrics[~metrics["model"].str.contains(" - ")].copy()
    plt.figure(figsize=(8, 5))
    plt.bar(main_metrics["model"], main_metrics["RMSE"])
    plt.title("Model comparison on time-based holdout set")
    plt.ylabel("RMSE (S$)")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(outdir / "05_model_rmse.png", dpi=200)
    plt.close()


def write_summary(best_name, metrics, sim, outdir: Path):
    best_metrics = metrics[metrics["model"] == best_name].iloc[0]
    cat_summaries = []
    for cat, d in sim.groupby("vehicle_class"):
        base = d[d["quota_increment"] == 0].iloc[0]
        plus_100 = d[d["quota_increment"] == 100].iloc[0]
        plus_300 = d[d["quota_increment"] == 300].iloc[0]
        cat_summaries.append(
            f"- {cat}: baseline predicted premium S${base['predicted_premium']:,.0f}; "
            f"+100 quota -> S${plus_100['predicted_premium']:,.0f} "
            f"({plus_100['price_change_vs_base']:,.0f}); "
            f"+300 quota -> S${plus_300['predicted_premium']:,.0f} "
            f"({plus_300['price_change_vs_base']:,.0f})."
        )

    text = f"""# COE Price Prediction & Quota Elasticity

Selected model: {best_name} (lowest RMSE on the 2021-2025 time-based holdout).

- MAE: S${best_metrics['MAE']:,.0f}
- RMSE: S${best_metrics['RMSE']:,.0f}
- MAPE: {best_metrics['MAPE']:.1f}%
- R2: {best_metrics['R2']:.2f}

Quota alone doesn't tell the full story. What the model actually leans on is
demand pressure - bids received per unit of quota. Add quota while demand
stays flat and the bid-to-quota ratio drops, which is what pulls predicted
premiums down in the simulation below.

Latest-round simulation:

{chr(10).join(cat_summaries)}

Caveat: this is a model-based sensitivity check, not a causal estimate.
Announcing more quota could itself change bidding behaviour, which this
setup can't capture. A proper causal read would need something like a
quasi-experimental design around past quota announcement shocks.
"""
    (outdir / "analysis_summary.md").write_text(text, encoding="utf-8")


def main():
    outdir = Path("outputs")

    df = load_and_clean()
    best_name, best_model, metrics, train_df, test_df, test_out, features = train_models(df)
    sim = run_policy_simulation(df, best_model, features)

    outdir.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(outdir / "model_metrics.csv", index=False)
    test_out.to_csv(outdir / "holdout_predictions.csv", index=False)
    sim.to_csv(outdir / "policy_simulation.csv", index=False)
    plot_outputs(df, test_out, sim, metrics, outdir)
    write_summary(best_name, metrics, sim, outdir)

    print("Done.")
    print(f"Best model: {best_name}")
    print(metrics[~metrics["model"].str.contains(" - ")].to_string(index=False))
    print(f"Outputs saved to: {outdir.resolve()}")


In [2]:
main()

Done.
Best model: Ridge baseline
            model         MAE         RMSE       R2     MAPE
   Ridge baseline 4108.987802  6460.280436 0.807118 3.963109
Gradient Boosting 7985.893799 10535.128698 0.487058 7.112033
    Random Forest 7815.447634 10570.944001 0.483564 7.035893
Outputs saved to: C:\Users\dodob\Desktop\Accessment\govtech_section1_q1\section-1-question-2\codes\outputs


## Interpretation

The simulation holds bids received constant when quota changes, so it's a policy sensitivity check rather than a causal estimate.